In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Load dataset
df = pd.read_csv('attached_assets/UCI_Credit_Card_1764961982346.csv')

In [ ]:
# --- Data Preprocessing & Cleaning ---

# 1. Initial Exploration
print("--- Data Info ---")
print(df.info())
print("\n--- Data Head ---")
print(df.head())
print("\n--- Data Shape ---")
print(df.shape)
print("\n--- Data Description ---")
print(df.describe())

# 2. Renaming Variables (if needed)
# Example: Renaming PAY_0 to PAY_1 for consistency if needed, though UCI dataset uses PAY_0
# df.rename(columns={'PAY_0': 'PAY_1'}, inplace=True)

# 3. Missing Values Analysis
print("\n--- Missing Values Before Imputation ---")
print(df.isnull().sum())

# Handle missing values (Numerical)
# Using SimpleImputer to fill with median for robust imputation
imputer = SimpleImputer(strategy='median')
cols_to_impute = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

# 4. Duplicate Values
print("\n--- Duplicate Values ---")
duplicates = df.duplicated().sum()
print(f"Found {duplicates} duplicates")
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates dropped.")

# 5. Outlier Analysis (LIMIT_BAL)
print("\n--- Outlier Analysis (LIMIT_BAL) ---")
Q1 = df['LIMIT_BAL'].quantile(0.25)
Q3 = df['LIMIT_BAL'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['LIMIT_BAL'] < lower_bound) | (df['LIMIT_BAL'] > upper_bound)]
print(f"Number of outliers detected in 'LIMIT_BAL': {len(outliers)}")

# 6. Statistical Metrics (LIMIT_BAL)
print("\n--- Statistical Metrics (LIMIT_BAL) ---")
stats = {
    'min': df['LIMIT_BAL'].min(),
    'q1': Q1,
    'median': df['LIMIT_BAL'].median(),
    'q3': Q3,
    'max': df['LIMIT_BAL'].max(),
    'std_dev': df['LIMIT_BAL'].std(),
    'iqr': IQR
}
print(stats)

# 7. Normalization
# StandardScaler (Z-score)
scaler = StandardScaler()
df['LIMIT_BAL_scaled'] = scaler.fit_transform(df[['LIMIT_BAL']])

# MinMaxScaler
minmax = MinMaxScaler()
df['LIMIT_BAL_normalized'] = minmax.fit_transform(df[['LIMIT_BAL']])

print("\n--- Normalized Data Sample ---")
print(df[['LIMIT_BAL', 'LIMIT_BAL_scaled', 'LIMIT_BAL_normalized']].head())

In [ ]:
# --- Analysis & Visualization ---

# 1. Default Rate
default_rate = df['default.payment.next.month'].value_counts(normalize=True) * 100
print("\n--- Tasa de Incumplimiento (Default) ---")
print(default_rate)

# 2. Limit Balance Distribution by Default Status
print("\n--- Distribución del Límite de Crédito por Estado de Default ---")
print(df.groupby('default.payment.next.month')['LIMIT_BAL'].describe())

# 3. Correlation Matrix
cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1', 'default.payment.next.month']
corr = df[cols].corr()
print("\n--- Matriz de Correlación ---")
print(corr)

# Generate Plots
sns.set_theme(style="whitegrid")

# Plot 1: Default Count
plt.figure(figsize=(6, 6))
sns.countplot(x='default.payment.next.month', data=df, palette='pastel')
plt.title('Distribución de Clientes: Cumplimiento (0) vs Default (1)')
plt.xlabel('Estado (0=No Default, 1=Default)')
plt.ylabel('Cantidad de Clientes')
plt.savefig('client/public/credit_default_count.png')

# Plot 2: Limit Balance Distribution (Boxplot)
plt.figure(figsize=(8, 6))
sns.boxplot(x='default.payment.next.month', y='LIMIT_BAL', data=df, palette='Set2')
plt.title('Distribución de Límite de Crédito por Estado')
plt.xlabel('Estado (0=No Default, 1=Default)')
plt.ylabel('Límite de Crédito ($)')
plt.savefig('client/public/credit_limit_dist.png')

# Plot 3: Age Distribution by Default
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='AGE', hue='default.payment.next.month', kde=True, element="step", palette='husl')
plt.title('Distribución de Edad por Riesgo de Default')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.savefig('client/public/credit_age_risk.png')

# Plot 4: Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlación entre Variables Clave')
plt.tight_layout()
plt.savefig('client/public/credit_correlation.png')

print("\nCredit Analysis complete. Images saved to client/public/")